# Story films · four short films, one point each

Four short animations for the capstone presentation. Each one makes a single point, with every element labelled and nothing else on screen:

| Film | The point | Length |
|---|---|---|
| **S1 · The bar to beat** | every model against the official forecast (SMARD), and the ensemble against all of them | 12 s |
| **S2 · Month by month** | in how many months of the test year the ensemble beats SMARD | 12 s |
| **S3 · Where the gain is** | SMARD against the ensemble in ordinary hours, below zero and in the extremes | 14 s |
| **S4 · Catching risk days** | risk days caught, missed and falsely flagged the evening before | 16 s |

**Inputs.** This notebook computes nothing new. It reads what the other notebooks wrote to `data/`:

| File | Written by | Used for |
|---|---|---|
| `data/models/ensemble_scoreboard.csv` | `ensemble-claude.ipynb` §6 | S1 average errors, S3 errors by situation |
| `data/models/ensemble_forecast_errors_hourly.csv`, `ensemble_members_hourly.csv` | `ensemble-claude.ipynb` §6 | S2 monthly errors on the common hours |
| `data/metrics/smard_forecast_errors_hourly.csv` | the SMARD baseline notebook | S2 SMARD's monthly errors |
| `data/films/film_inputs.joblib` | `ensemble-claude.ipynb` §8 | which ensemble is shown, the test year |
| `data/risk_classification/model_risk_labels_daily.csv` | `visualization-02-classification-risk-labels.ipynb` | S4 caught, missed, false alarms |

**Fair test.** All numbers are on the test year only. The ensemble shown was picked on the validation year (lowest out-of-fold error), and the test year was scored once.

Run top to bottom. §3.2 shows the last frame of each film in seconds; §4 renders the mp4s (about 1–2 minutes per film at 1920 × 1080); §5 plays them here.

## 1 Setup

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, Video, display

# Settings
FILMS_TO_RENDER = ["story_S1_the_bar_to_beat", "story_S2_month_by_month",
                   "story_S3_where_the_gain_is", "story_S4_catching_risk_days"]
RENDER_FILMS = True    # True: write the mp4s that are missing or out of date; False: §3 previews only
FORCE_RENDER = False   # True: render again even when an mp4 is up to date
HD = True              # True: 1920 × 1080; False: 1280 × 720 (renders about twice as fast)
HOLD_SECONDS = 2       # the last frame stays on screen this long
SHOW_FILMS = "auto"    # §5: "link" plays the mp4s from disk, "embed" stores them in the notebook,
                       # "auto": embed in VS Code (which cannot play linked files), link elsewhere; None: off

# Find the repository's data/ folder: the first folder, from here upwards, whose data/ holds smard.csv.
# Set the environment variable GRIDSTRESS_DATA to point at a data/ folder elsewhere.
candidates = [Path(os.environ["GRIDSTRESS_DATA"])] if os.environ.get("GRIDSTRESS_DATA") else []
candidates += [folder / "data" for folder in [Path.cwd(), *Path.cwd().parents]]
DATA_DIR = next((path for path in candidates if (path / "smard.csv").is_file()), None)
if DATA_DIR is None:
    raise RuntimeError(
        f"No data/smard.csv found in {Path.cwd()} or any parent folder. Run the notebook from inside the "
        "repository, or set GRIDSTRESS_DATA to the data/ folder that holds smard.csv."
    )
FILM_DIR = DATA_DIR / "films"  # data/ is gitignored
FILM_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "scoreboard": (DATA_DIR / "models" / "ensemble_scoreboard.csv", "ensemble-claude.ipynb (§6)"),
    "ensembles": (DATA_DIR / "models" / "ensemble_forecast_errors_hourly.csv", "ensemble-claude.ipynb (§6)"),
    "members": (DATA_DIR / "models" / "ensemble_members_hourly.csv", "ensemble-claude.ipynb (§6)"),
    "smard": (DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv", "the SMARD baseline notebook"),
    "film_inputs": (DATA_DIR / "films" / "film_inputs.joblib", "ensemble-claude.ipynb (§8)"),
    "risk": (DATA_DIR / "risk_classification" / "model_risk_labels_daily.csv",
             "visualization-02-classification-risk-labels.ipynb"),
}
missing = [f"  {path.relative_to(DATA_DIR.parent).as_posix()}  ← run {maker}"
           for key, (path, maker) in SOURCES.items() if key != "risk" and not path.exists()]
if missing:
    raise FileNotFoundError("These inputs are missing:\n" + "\n".join(missing))
HAS_RISK = SOURCES["risk"][0].exists()
if not HAS_RISK:
    print(f"{SOURCES['risk'][0].name} not found: S4 is skipped. Run {SOURCES['risk'][1]} to make it.")
for key, (path, _) in SOURCES.items():
    if path.exists():
        print(f"{path.relative_to(DATA_DIR.parent).as_posix():62s} saved {datetime.fromtimestamp(path.stat().st_mtime):%d %b %Y %H:%M}")

## 2 The drawing code

One function per film, `draw(fig, t, STORY)`, with `t` running from 0 (empty page) to 1 (the full picture). The headline is the takeaway and is worked out from the numbers, so it stays true if they change. Colours are the team palette of `streamlit/viz_helpers.py`.

In [ ]:
# Story films: one idea per film, every element labelled, nothing else on screen.
# Input: STORY, a small dict of numbers (built from the exports in story-films.ipynb).

# The team palette (streamlit/viz_helpers.py)
INK, MUTED, FAINT, GRID, CARD = "#1B3C65", "#5C6275", "#A3A7B6", "#E4E6EE", "#F0F1F5"
TEAL, VERMILLION, AMBER = "#12989A", "#C8553D", "#E7A118"
SMARD_C, ENSEMBLE_C = "#6B6E80", "#7A2A06"
MODEL_C = {
    "random_forest_hybrid": "#12989A", "lgbm_direct": "#2E64B0", "linear_direct": "#C27C00",
    "xgb_hybrid": "#9A5FA8", "lgbm_hybrid": "#C8553D", "xgb_direct": "#5566C2",
}
SIZE, DPI, FPS = (12.8, 7.2), 150, 24  # 1920 × 1080 at 24 frames per second


def ease(t):
    t = min(max(t, 0.0), 1.0)
    return 1 - (1 - t) ** 3


def seg(t, a, b):
    return min(max((t - a) / (b - a), 0.0), 1.0)


def frame(fig, headline, subline, source, t, headline_in=(0.0, 0.08)):
    """White page, a small kicker, the headline (the takeaway), one subline, a quiet source line."""
    fig.clf()
    fig.patch.set_facecolor("white")
    a = ease(seg(t, *headline_in))
    fig.text(0.06, 0.925, "GRIDSTRESS", color=TEAL, fontsize=11, fontweight="bold", alpha=a)
    fig.text(0.06, 0.855, headline, color=INK, fontsize=25, fontweight="bold", alpha=a)
    fig.text(0.06, 0.805, subline, color=MUTED, fontsize=13, alpha=a)
    fig.text(0.06, 0.035, source, color=FAINT, fontsize=9.5)


def bare(ax):
    """An axes with nothing but what the film draws on it."""
    ax.set_axis_off()
    return ax


def pct(value, digits=1):
    return f"{'+' if value > 0 else '−' if value < 0 else ''}{abs(value):.{digits}f} %"


# ---------- S1 · the bar to beat ----------
def s1(fig, t, S):
    rows = S["field"]  # [(label, colour, mae, kind)], kind in smard / model / ensemble, in drawing order
    smard = next(r for r in rows if r[3] == "smard")[2]
    ens = next(r for r in rows if r[3] == "ensemble")
    better = 100 * (1 - ens[2] / smard)
    models = [r for r in rows if r[3] == "model"]
    beat = sum(r[2] < smard for r in models)
    best = ens[2] <= min(r[2] for r in models)
    lead = "Every model beats the official forecast." if beat == len(models) else \
        f"{beat} of {len(models)} models beat the official forecast."
    frame(fig, lead + (" Combined, they beat it most." if best and ens[2] < smard else ""),
          f"average error per hour on the test year {S['period']} · lower is better",
          f"MAE in MWh per hour on {S['hours']:,} common hours · SMARD = the official day-ahead forecast "
          "(Bundesnetzagentur)", t)
    ax = bare(fig.add_axes([0.25, 0.12, 0.62, 0.62]))
    n, top = len(rows), max(r[2] for r in rows) * 1.18
    ax.set_xlim(0, top)
    ax.set_ylim(n - 0.4, -0.7)
    for i, (label, colour, mae, kind) in enumerate(rows):
        start = 0.08 if kind == "smard" else 0.30 if kind == "model" else 0.55
        grow = ease(seg(t, start + (0.02 * i if kind == "model" else 0), start + 0.14 + (0.02 * i if kind == "model" else 0)))
        height = 0.72 if kind != "model" else 0.56
        alpha = 1.0 if kind != "model" else 0.55
        ax.barh(i, mae * grow, height=height, color=colour, alpha=alpha, linewidth=0)
        if grow > 0:
            weight = "bold" if kind != "model" else "normal"
            ax.text(-top * 0.015, i, label, ha="right", va="center", fontsize=14 if kind != "model" else 12,
                    color=INK if kind != "model" else MUTED, fontweight=weight, alpha=min(1, grow * 2))
        if grow >= 1:
            ax.text(mae + top * 0.012, i, f"{mae:,.0f}", va="center", fontsize=14 if kind != "model" else 12,
                    color=colour if kind != "model" else MUTED, fontweight="bold" if kind != "model" else "normal")
    wall = ease(seg(t, 0.22, 0.30))
    if wall > 0:
        ax.plot([smard, smard], [-0.5, -0.5 + wall * n], color=SMARD_C, linewidth=1.4, linestyle=(0, (6, 4)), zorder=0)
    shout = ease(seg(t, 0.75, 0.88))
    if shout > 0:
        i = rows.index(ens)
        ax.annotate("", xy=(ens[2], i + 0.55), xytext=(smard, i + 0.55),
                    arrowprops={"arrowstyle": "<->", "color": ENSEMBLE_C, "linewidth": 2, "alpha": shout})
        ax.text((smard + ens[2]) / 2, i + 0.95, f"{pct(-better)} error", ha="center", va="top", fontsize=19,
                fontweight="bold", color=ENSEMBLE_C, alpha=shout)


# ---------- S2 · month by month ----------
def s2(fig, t, S):
    months = S["months"]  # [(label, smard_mae, ensemble_mae)]
    skill = [100 * (1 - e / s) for _, s, e in months]
    won = sum(v > 0 for v in skill)
    frame(fig, f"Better than the official forecast in {won} of {len(months)} months",
          "error reduction of the ensemble against SMARD, per full month · above the line = better",
          "error reduction = 1 − ensemble MAE / SMARD MAE, per calendar month of the test year", t)
    ax = fig.add_axes([0.09, 0.15, 0.84, 0.56])
    bare(ax)
    n = len(months)
    hi = max(skill) * 1.25
    lo = min(min(skill), 0) - hi * 0.32  # room under the lowest bar for its value and the month names
    ax.set_xlim(-0.7, n - 0.3)
    ax.set_ylim(lo, hi)
    ax.axhline(0, color=SMARD_C, linewidth=1.6, linestyle=(0, (6, 4)))
    ax.text(n - 0.35, 0, "SMARD ", ha="right", va="bottom", fontsize=11, color=SMARD_C, fontweight="bold")
    shown = worse = 0
    for i, ((label, _, _), value) in enumerate(zip(months, skill)):
        grow = ease(seg(t, 0.10 + i * 0.06, 0.18 + i * 0.06))
        colour = TEAL if value > 0 else VERMILLION
        ax.bar(i, value * grow, width=0.64, color=colour, linewidth=0)
        ax.text(i, lo, label, ha="center", va="bottom", fontsize=12, color=MUTED)
        if grow >= 1:
            shown += value > 0
            worse += value <= 0
            ax.text(i, value + (hi - lo) * (0.025 if value > 0 else -0.025), pct(value, 0), ha="center",
                    va="bottom" if value > 0 else "top", fontsize=13, fontweight="bold", color=colour)
    counter = ease(seg(t, 0.15, 0.25))
    fig.text(0.93, 0.74, f"so far: {shown} better · {worse} worse", ha="right", fontsize=15, color=MUTED, fontweight="bold", alpha=counter)


# ---------- S3 · where the gain is ----------
def s3(fig, t, S):
    rows = S["scenarios"]  # [(name, hours, smard_mae, ensemble_mae)]
    change_of = {name: 100 * (e / s_ - 1) for name, _, s_, e in rows[1:]}
    top_name = min(change_of, key=change_of.get)
    stress = ("Below zero", "Highest 1 %")
    headline = ("The gains are largest where the grid is under stress" if top_name in stress
                else f"The ensemble gains most in the {top_name.split(chr(10))[0].lower()} hours")
    frame(fig, headline,
          "average error per hour, SMARD against the ensemble, by situation · lower is better",
          "situations by the actual residual load: ordinary = middle 50 % of hours, lowest / highest 1 % of hours, below zero", t)
    ax = bare(fig.add_axes([0.07, 0.22, 0.86, 0.47]))
    n = len(rows)
    top = max(max(s, e) for _, _, s, e in rows) * 1.22
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_ylim(-top * 0.20, top)
    for i, (name, hours, smard, ens) in enumerate(rows):
        start = 0.08 + i * 0.13
        g1, g2 = ease(seg(t, start, start + 0.06)), ease(seg(t, start + 0.04, start + 0.10))
        ax.bar(i - 0.19, smard * g1, width=0.36, color=SMARD_C, alpha=0.85, linewidth=0)
        ax.bar(i + 0.19, ens * g2, width=0.36, color=ENSEMBLE_C, linewidth=0)
        if g1 > 0:
            title, _, detail = name.partition("\n")
            detail = detail.strip("()")
            ax.text(i, -top * 0.04, title, ha="center", va="top", fontsize=14, color=INK, fontweight="bold", alpha=g1)
            ax.text(i, -top * 0.12, f"{detail + ' · ' if detail else ''}{hours:,} hours", ha="center", va="top",
                    fontsize=11, color=MUTED, alpha=g1)
        if g1 >= 1:
            ax.text(i - 0.19, smard + top * 0.012, f"{smard:,.0f}", ha="center", va="bottom", fontsize=11, color=SMARD_C)
        if g2 >= 1:
            change = 100 * (ens / smard - 1)
            ax.text(i + 0.19, ens + top * 0.012, f"{ens:,.0f}", ha="center", va="bottom", fontsize=11, color=ENSEMBLE_C)
            strong = change <= -10
            ax.text(i, max(smard, ens) + top * 0.09, pct(change, 0), ha="center", va="bottom", fontsize=20 if strong else 16,
                    fontweight="bold", color=TEAL if strong else MUTED)
    legend = ease(seg(t, 0.04, 0.12))
    for k, (label, colour) in enumerate([("SMARD, the official forecast", SMARD_C), ("our ensemble", ENSEMBLE_C)]):
        x = 0.06 + k * 0.23
        fig.patches.append(mpatches.Rectangle((x, 0.735), 0.014, 0.022, transform=fig.transFigure, color=colour, alpha=legend))
        fig.text(x + 0.02, 0.746, label, fontsize=12, color=INK, va="center", alpha=legend)
    weak = [r for r in rows if 100 * (r[3] / r[2] - 1) > -10]
    note = ease(seg(t, 0.82, 0.92))
    if weak and note > 0:
        names = " and ".join(r[0].lower() for r in weak)
        fig.text(0.5, 0.095, f"Honest limit: in the {names} the ensemble gains little; averaging pulls the deepest lows up.",
                 ha="center", fontsize=12.5, color=MUTED, alpha=note)


# ---------- S4 · catching risk days ----------
def s4(fig, t, S):
    R = S["risk"]
    ours = {v: sum(R[k]["model"][v] for k in ("high", "low")) for v in ("caught", "false alarm")}
    theirs = {v: sum(R[k]["smard"][v] for k in ("high", "low")) for v in ("caught", "false alarm")}
    if ours["caught"] > theirs["caught"]:
        headline = ("We catch more risk days, with a few more false alarms" if ours["false alarm"] > theirs["false alarm"]
                    else "We catch more risk days, without more false alarms")
    else:
        headline = "Risk days caught: our forecast against the official one"
    frame(fig, headline,
          "test-year days the forecast saw coming the evening before · one square = one actual risk day",
          f"our forecast: {R['high']['model_label']} for high risk, {R['low']['model_label']} for below zero (each the best model there) · "
          "labels from risk-definition.ipynb", t)
    panels = [("high", "High-risk days", "an hour above the top 1 % of the past year", 0.06, 0.40),
              ("low", "Below-zero days", "an hour with more wind and solar than demand", 0.50, 0.44)]
    for p, (key, title, sub, x0, width) in enumerate(panels):
        r = R[key]
        a = ease(seg(t, 0.06 + p * 0.40, 0.14 + p * 0.40))
        fig.text(x0, 0.70, f"{title}: {r['days']}", fontsize=19, fontweight="bold", color=INK, alpha=a)
        fig.text(x0, 0.665, sub, fontsize=11.5, color=MUTED, alpha=a)
        per_row = 12 if r["days"] <= 24 else 28
        rows_needed = int(np.ceil(r["days"] / per_row))
        cell = min(width / per_row, 0.0185 if per_row > 12 else 0.03)
        for k, (who, label, colour) in enumerate([("smard", "SMARD", SMARD_C), ("model", "Ours", ENSEMBLE_C)]):
            y_top = 0.58 - k * (rows_needed * cell * 1.78 + 0.14)
            fill = seg(t, 0.12 + p * 0.40 + k * 0.10, 0.30 + p * 0.40 + k * 0.10)
            if a <= 0:
                continue
            fig.text(x0, y_top + 0.012, label, fontsize=14, fontweight="bold", color=colour, alpha=a)
            caught = r[who]["caught"]
            for d in range(r["days"]):
                row, col = divmod(d, per_row)
                x, y = x0 + col * cell, y_top - 0.03 - row * cell * 1.78
                shown = d < int(r["days"] * fill + 0.999) and fill > 0
                is_caught = d < caught
                face = TEAL if (shown and is_caught) else "white"
                edge = TEAL if is_caught else VERMILLION
                fig.patches.append(mpatches.Rectangle((x, y - cell * 1.5), cell * 0.82, cell * 1.5, transform=fig.transFigure,
                                                      facecolor=face if shown else "white",
                                                      edgecolor=(edge if shown else GRID), linewidth=1.3 if shown else 0.8))
            if fill >= 1:
                y_text = y_top - 0.03 - rows_needed * cell * 1.78 - 0.012
                fig.text(x0, y_text, f"{caught} caught", fontsize=13, fontweight="bold", color=TEAL, va="top")
                fig.text(x0 + 0.115, y_text, f"{r[who]['missed']} missed", fontsize=13, fontweight="bold", color=VERMILLION, va="top")
                fig.text(x0 + 0.23, y_text, f"{r[who]['false alarm']} false alarms", fontsize=13, color=AMBER, fontweight="bold", va="top")
    key = ease(seg(t, 0.85, 0.95))
    if key > 0:
        for k, (label, face, edge) in enumerate([("caught: flagged the evening before", TEAL, TEAL),
                                                 ("missed: risk day not flagged", "white", VERMILLION)]):
            x = 0.06 + k * 0.30
            fig.patches.append(mpatches.Rectangle((x, 0.085), 0.011, 0.022, transform=fig.transFigure, facecolor=face,
                                                  edgecolor=edge, linewidth=1.3, alpha=key))
            fig.text(x + 0.017, 0.096, label, fontsize=11.5, color=INK, va="center", alpha=key)
        fig.text(0.66, 0.096, "false alarm: flagged, but no risk hour came", fontsize=11.5, color=AMBER, va="center", alpha=key)


FILMS = {
    "story_S1_the_bar_to_beat": (s1, 12),
    "story_S2_month_by_month": (s2, 12),
    "story_S3_where_the_gain_is": (s3, 14),
    "story_S4_catching_risk_days": (s4, 16),
}


def code_print(code):
    """The bytecode and constants of a function, nested code included: changes when the drawing code changes."""
    return repr(code.co_code) + "".join(code_print(c) if hasattr(c, "co_code") else repr(c) for c in code.co_consts)


DRAW_SOURCE = "".join(code_print(f.__code__) for f in (frame, s1, s2, s3, s4))  # for the render fingerprint of §4

## 3 The numbers

Everything the films show, collected into one small dictionary, `STORY`. The table under the cell lists every number that appears on screen.

- **Common hours**: the test-year hours where the actual and every forecast exist, as in the ensemble notebook.
- **Full months** only: the partial first and last month of the test year stay out of S2.
- **Risk days** follow the `any` rule of `risk-definition.ipynb`: a day is a risk day when one hour crosses that day's line. *High risk* uses the rolling top-1 % line, *below zero* the zero line.

In [ ]:
LABEL = {
    "random_forest_hybrid": "Random forest hybrid", "lgbm_direct": "LightGBM direct", "linear_direct": "Ridge direct",
    "xgb_hybrid": "XGBoost hybrid", "lgbm_hybrid": "LightGBM hybrid", "xgb_direct": "XGBoost direct",
    "sarimax_fourier": "SARIMAX + Fourier",
}
NOT_MEMBERS = {"smard", "seasonal_naive"}
STAMP = "%Y-%m-%d %H:%M:%S"

# Which ensemble, which split, which test year: from the film inputs of ensemble-claude.ipynb §8
INPUTS = joblib.load(SOURCES["film_inputs"][0])
FILM = INPUTS.get("film", {})
HERO_METHOD = FILM.get("hero_method", "regime_weighted")
HERO_KEY = f"ensemble_{HERO_METHOD}"
HERO_LABEL = FILM.get("hero_label", HERO_METHOD.replace("_", "-").capitalize())
SPLIT = FILM.get("split", "rolling")
TEST_FIRST, TEST_LAST = (pd.Timestamp(day).normalize() for day in FILM["test"])
TEST_HOURS = pd.date_range(TEST_FIRST, TEST_LAST + pd.Timedelta(hours=23), freq="h")

# S1 and S3: the scoreboard of the test year, exactly as ensemble-claude.ipynb scored it
board = pd.read_csv(SOURCES["scoreboard"][0])


def scored(model, split, table, metric):
    """(value, hours) of one scoreboard row."""
    row = board[(board["model"] == model) & (board["split_method"] == split)
                & (board["table"] == table) & (board["metric"] == metric)]
    if row.empty:
        raise KeyError(f"no scoreboard row for {model} / {split} / {table} / {metric}: re-run ensemble-claude.ipynb")
    return float(row["value"].iloc[0]), int(row["count"].iloc[0])


smard_mae, hours = scored("smard", "none", "accuracy", "MAE")
hero_mae, _ = scored(HERO_KEY, SPLIT, "accuracy", "MAE")
member_keys = [key for key in board.loc[(board["split_method"] == SPLIT) & (board["table"] == "accuracy"), "model"].unique()
               if not key.startswith("ensemble_") and key not in NOT_MEMBERS]
members = sorted(((LABEL.get(key, key), MODEL_C.get(key, FAINT), scored(key, SPLIT, "accuracy", "MAE")[0], "model")
                  for key in member_keys), key=lambda row: -row[2])
field = [("SMARD · official forecast", SMARD_C, smard_mae, "smard")] + members + [("Our ensemble", ENSEMBLE_C, hero_mae, "ensemble")]

scenarios = [("All hours", hours, smard_mae, hero_mae)]
for name, bin_ in [("Ordinary\n(middle 50 %)", "ordinary"), ("Below zero", "below_zero"),
                   ("Lowest 1 %", "low_extreme"), ("Highest 1 %", "high_extreme")]:
    smard_bin, bin_hours = scored("smard", "none", "extremes", f"MAE_{bin_}_by_actual")
    hero_bin, _ = scored(HERO_KEY, SPLIT, "extremes", f"MAE_{bin_}_by_actual")
    scenarios.append((name, bin_hours, smard_bin, hero_bin))

# S2: monthly errors from the hourly files, on the common hours of the test year
def hourly(path, usecols):
    table = pd.read_csv(path, usecols=usecols)
    table["timestamp"] = pd.to_datetime(table["timestamp"], format=STAMP)
    return table


ens = hourly(SOURCES["ensembles"][0], ["timestamp", "model", "split_method", "forecast", "err_residual_load"])
hero_err = ens[(ens["model"] == HERO_KEY) & (ens["split_method"] == SPLIT)].set_index("timestamp")["err_residual_load"]
member_rows = hourly(SOURCES["members"][0], ["timestamp", "model", "split_method", "forecast"])
complete = member_rows.pivot_table(index="timestamp", columns=["model", "split_method"], values="forecast", aggfunc="first")
smard_err = hourly(SOURCES["smard"][0], ["timestamp", "err_residual_load"]).set_index("timestamp")["err_residual_load"]

common = TEST_HOURS.intersection(complete.dropna().index).intersection(hero_err.dropna().index) \
    .intersection(smard_err.dropna().index)
errors = pd.DataFrame({"smard": smard_err.reindex(common).abs(), "hero": hero_err.reindex(common).abs()})
month_first = pd.Period(TEST_FIRST, "M") if TEST_FIRST.day == 1 else pd.Period(TEST_FIRST, "M") + 1
month_last = pd.Period(TEST_LAST, "M") if TEST_LAST.is_month_end else pd.Period(TEST_LAST, "M") - 1
monthly = errors.groupby(errors.index.to_period("M")).mean().loc[month_first:month_last]
months = [(period.strftime("%b\n%y"), float(row["smard"]), float(row["hero"])) for period, row in monthly.iterrows()]

# S4: risk days from visualization-02, on the test days
risk = None
if HAS_RISK:
    daily = pd.read_csv(SOURCES["risk"][0])
    daily["date"] = pd.to_datetime(daily["date"], format="%Y-%m-%d")
    daily = daily[(daily["date"] >= TEST_FIRST) & (daily["date"] <= TEST_LAST)]
    flag = lambda column: daily[column].map({True: True, False: False, "True": True, "False": False}).astype("boolean")
    if "evaluable" in daily:
        daily = daily[flag("evaluable").fillna(False).to_numpy()]
    risk = {}
    for kind, rule, model_column in [("high", "high_risk_rolling_any", "high_model"), ("low", "low_risk_zero_any", "low_model")]:
        actual = flag(f"actual_{rule}")
        known = actual.notna()
        actual = actual[known].astype(bool)
        tally = {}
        for who, prefix in [("model", "model"), ("smard", "smard")]:
            said = flag(f"{prefix}_{rule}")[known].fillna(False).astype(bool)
            tally[who] = {"caught": int((actual & said).sum()), "missed": int((actual & ~said).sum()),
                          "false alarm": int((~actual & said).sum())}
        model_key = daily.loc[known, model_column].mode().iloc[0] if model_column in daily else "model"
        risk[kind] = {"days": int(actual.sum()), "model_label": LABEL.get(model_key, model_key), **tally}

STORY = {"period": f"{TEST_FIRST.day} {TEST_FIRST:%b %Y} – {TEST_LAST.day} {TEST_LAST:%b %Y}", "hours": hours, "field": field,
         "months": months, "scenarios": scenarios, "risk": risk, "hero_label": HERO_LABEL}

# Every number that appears on screen
print(f"ensemble shown: {HERO_LABEL} ({SPLIT}) · test year {STORY['period']} · {hours:,} common hours "
      f"(here: {len(common):,})\n")
print("S1 · average error per hour (MWh)")
for label, _, mae, _ in field:
    print(f"   {label:28s} {mae:7,.0f}")
print("\nS2 · per full month: SMARD → ensemble (MWh), error reduction")
for label, s, e in months:
    print(f"   {label.replace(chr(10), ' '):8s} {s:6,.0f} → {e:6,.0f}   {100 * (1 - e / s):+5.0f} %")
print("\nS3 · by situation: hours, SMARD → ensemble (MWh)")
for name, n, s, e in scenarios:
    print(f"   {name.replace(chr(10), ' '):26s} {n:6,} h  {s:6,.0f} → {e:6,.0f}   {100 * (e / s - 1):+5.0f} %")
if risk:
    print("\nS4 · risk days: caught / missed / false alarms")
    for kind, r in risk.items():
        print(f"   {kind:5s} {r['days']:4d} days · ours ({r['model_label']}) {r['model']['caught']} / {r['model']['missed']} / "
              f"{r['model']['false alarm']} · SMARD {r['smard']['caught']} / {r['smard']['missed']} / {r['smard']['false alarm']}")

### 3.1 Self-check

The hourly files must give the scoreboard's numbers back, so the monthly bars of S2 come from the same hours as S1 and S3.

In [ ]:
check_smard, check_hero = errors["smard"].mean(), errors["hero"].mean()
for name, mine, theirs in [("SMARD", check_smard, smard_mae), (HERO_LABEL, check_hero, hero_mae)]:
    gap = 100 * abs(mine / theirs - 1)
    print(f"{name:20s} MAE from the hourly files {mine:7,.1f} · scoreboard {theirs:7,.1f} · gap {gap:.2f} %")
    assert gap < 1.0, f"{name}: the hourly files and the scoreboard disagree; re-run ensemble-claude.ipynb top to bottom"
assert abs(len(common) - hours) <= 0.01 * hours, f"common hours {len(common)} against the scoreboard's {hours}"
assert len(months) >= 1, "no full month in the test year"
if risk:
    for kind, r in risk.items():
        for who in ("model", "smard"):
            assert r[who]["caught"] + r[who]["missed"] == r["days"], (kind, who)
print("all checks passed")

### 3.2 Preview · the last frame of each film

In [ ]:
FILMS_HERE = {name: spec for name, spec in FILMS.items() if name != "story_S4_catching_risk_days" or risk}
fig_sheet, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, (name, (draw, _)) in zip(axes.flat, FILMS_HERE.items()):
    fig = plt.figure(figsize=SIZE, dpi=60)
    draw(fig, 1.0, STORY)
    fig.canvas.draw()
    ax.imshow(np.asarray(fig.canvas.buffer_rgba()))
    plt.close(fig)
for ax in axes.flat:
    ax.set_axis_off()
fig_sheet.tight_layout()
plt.show()

## 4 Render

In [ ]:
FILM_DPI = DPI if HD else 100  # SIZE is 12.8 × 7.2 inches: 150 dpi = 1920 × 1080, 100 dpi = 1280 × 720


def ffmpeg_path():
    """A system ffmpeg on the PATH, or the one shipped with imageio-ffmpeg."""
    import shutil
    found = shutil.which("ffmpeg")
    if found:
        return found
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError as error:
        raise RuntimeError("No ffmpeg found: install it, or run  pip install imageio-ffmpeg") from error


def fingerprint(name):
    """Changes when the numbers, the drawing code or the size change: then the mp4 is out of date."""
    source = DRAW_SOURCE + json.dumps(STORY, default=str, sort_keys=True) + f"{name}{FILM_DPI}{HOLD_SECONDS}{FPS}"
    return hashlib.sha256(source.encode()).hexdigest()[:16]


if RENDER_FILMS:
    from matplotlib.animation import FFMpegWriter, FuncAnimation
    matplotlib.rcParams["animation.ffmpeg_path"] = ffmpeg_path()
    for name in FILMS_TO_RENDER:
        if name not in FILMS_HERE:
            print(f"{name}: skipped (its inputs are missing)")
            continue
        path, stamp = FILM_DIR / f"{name}.mp4", FILM_DIR / f"{name}.json"
        if not FORCE_RENDER and path.exists() and stamp.exists() and json.loads(stamp.read_text()).get("fingerprint") == fingerprint(name):
            print(f"{name}: up to date")
            continue
        draw, seconds = FILMS_HERE[name]
        frames, hold = seconds * FPS, HOLD_SECONDS * FPS
        started = time.perf_counter()
        fig = plt.figure(figsize=SIZE, dpi=FILM_DPI)
        movie = FuncAnimation(fig, lambda k: draw(fig, min(k / (frames - 1), 1.0), STORY), frames=frames + hold)
        movie.save(path, writer=FFMpegWriter(fps=FPS, bitrate=6000, extra_args=["-pix_fmt", "yuv420p"]), dpi=FILM_DPI)
        plt.close(fig)
        stamp.write_text(json.dumps({"fingerprint": fingerprint(name), "rendered": datetime.now().isoformat(timespec="seconds"),
                                     "ensemble": HERO_LABEL, "period": STORY["period"]}))
        print(f"wrote {path.relative_to(DATA_DIR.parent).as_posix()}: {frames + hold} frames, "
              f"{(frames + hold) / FPS:.0f} s, {path.stat().st_size / 1e6:.1f} MB in {time.perf_counter() - started:.0f} s", flush=True)
else:
    print("RENDER_FILMS = False: nothing rendered (§3.2 shows the last frames)")

## 5 Watch

In [ ]:
if SHOW_FILMS not in ("auto", "link", "embed", None):
    raise ValueError(f'SHOW_FILMS must be "auto", "link", "embed" or None, not {SHOW_FILMS!r}')
in_vscode = any(key.startswith("VSCODE_") for key in os.environ)
show_mode = ("embed" if in_vscode else "link") if SHOW_FILMS == "auto" else SHOW_FILMS
TITLES = {"story_S1_the_bar_to_beat": "S1 · The bar to beat", "story_S2_month_by_month": "S2 · Month by month",
          "story_S3_where_the_gain_is": "S3 · Where the gain is", "story_S4_catching_risk_days": "S4 · Catching risk days"}
if show_mode:
    for name in FILMS_HERE:
        path = FILM_DIR / f"{name}.mp4"
        if not path.exists():
            print(f"{name}: not rendered yet (set RENDER_FILMS = True in §1)")
            continue
        display(Markdown(f"**{TITLES[name]}** · `{path.relative_to(DATA_DIR.parent).as_posix()}` · "
                         f"{path.stat().st_size / 1e6:.1f} MB"))
        if show_mode == "embed":
            display(Video(str(path), embed=True, width=880, html_attributes="controls loop"))
        else:
            # Jupyter serves files relative to the notebook's folder, so the link must be relative too
            display(Video(Path(os.path.relpath(path, Path.cwd())).as_posix(), width=880, html_attributes="controls loop"))
else:
    print("SHOW_FILMS = None: films are not played here")